# **Pathway Enrichment Analysis using g:Profiler**

# Import Libraries and Configurations

In [1]:
import json
import os
import sys
from urllib.parse import urljoin

import pandas as pd
import requests

# Add project root to Python's path
sys.path.append(os.path.abspath(os.path.join('..')))

from config import (
    NETWORK_DATA_DIRS,
    NETWORK_FILES,
)

# G:Profiler (GP) API endpoints of interest
GP_API_BASE_URL = 'https://biit.cs.ut.ee/gprofiler/api/'
GP_API_ENDPOINTS = {
    'data_versions': urljoin(GP_API_BASE_URL, 'util/data_versions'),
    'functional_profiling': urljoin(GP_API_BASE_URL, 'gost/profile'),
}

ORGANISM = 'hsapiens'

# Functions

In [2]:
def pathway_enrichment(df_genes):
    # List all genes/messenger RNAs in the network
    query = df_genes['id'].to_list()

    # Define the parameters for functional enrichment analysis
    parameters = {
        'organism': ORGANISM,
        'query': query,
        'sources': ['KEGG'],
        'user_threshold': 0.05,
        'all_results': False,
        'significance_threshold_method': 'fdr',
    }

    # Request analysis for the g:Profile API
    response = requests.post(
        url=GP_API_ENDPOINTS['functional_profiling'],
        json=parameters,
    )
    response.raise_for_status()

    # Convert the response content to a JSON
    json_response = json.loads(response.content.decode('utf-8'))

    # Transform the JSON response result into a DataFrame
    df_enrichment = pd.json_normalize(json_response['result'])

    # Recover the enriched genes/messenger RNAs
    df_enrichment['genes'] = df_enrichment['intersections'] \
        .apply(
            lambda col: [
                genes for genes, sources in zip(query, col) if sources
            ]
        )

    # Remove the unnecessary columns
    df_enrichment = df_enrichment[['native','name','p_value','genes']]

    return df_enrichment

In [3]:
def create_pathway_related_network(group):
    # Define the path to the directory related to the group
    group_dir = (group.lower()).replace(' ', '-')
    dir_path = NETWORK_DATA_DIRS['processed'][group_dir]

    # Create a node DataFrame for the microRNA-mRNA interaction network
    file_name = NETWORK_FILES['interaction-nodes']
    df_interaction_nodes = pd.read_csv(os.path.join(dir_path, file_name))
    
    # Pathway enrichment analysis of genes/messenger RNAs
    df_genes = df_interaction_nodes.query('type == "Messenger RNA"')
    df_enrichment = pathway_enrichment(df_genes=df_genes)

    # Create a DataFrame of pathway nodes
    df_pathways = df_enrichment[['native', 'name']] \
        .rename(columns={'native': 'id', 'name': 'label'})
    df_pathways['type'] = 'Pathway'

    # Create a DataFrame of edges between genes and pathways
    df_membership_edges = df_enrichment \
        .explode(column='genes', ignore_index=True) \
        [['genes', 'native', 'p_value']] \
        .rename(columns={
            'genes': 'source',
            'native': 'target',
            'p_value': 'qvalue',
        })
    
    # Create a DataFrame of enriched gene nodes
    df_enriched_genes = df_membership_edges[['source']] \
        .rename(columns={'source': 'id'}) \
        .drop_duplicates(ignore_index=True)
    df_enriched_genes['label'] = df_enriched_genes['id']
    df_enriched_genes['type'] = 'Messenger RNA'

    # Concatenate enriched gene and pathway nodes in a single DataFrame
    df_membership_nodes = pd.concat(
        objs=[df_enriched_genes, df_pathways],
        ignore_index=True
    )

    # Store the DataFrame of edges
    file_name = 'membership-network-edges.csv'
    df_membership_edges.to_csv(os.path.join(dir_path, file_name), index=False)

    # Store the DataFrame of nodes
    file_name = 'membership-network-nodes.csv'
    df_membership_nodes.to_csv(os.path.join(dir_path, file_name), index=False)
    
    return df_membership_edges, df_membership_nodes

# Data Versions Endpoint

In [4]:
# Request the API version and status to the endpoint
response = requests.get(
    url=GP_API_ENDPOINTS['data_versions'],
    json={'organism': ORGANISM}
)

# Print the response content
try:
    parsed = json.loads(response.content.decode('utf-8'))
    print(json.dumps(parsed, indent=3))
except json.JSONDecodeError:
    print('Response is not valid JSON:')
    print(response.content.decode('utf-8'))

{
   "biomart": "Ensembl",
   "biomart_version": "113",
   "display_name": "Human",
   "genebuild": "GRCh38.p14",
   "gprofiler_version": "e113_eg59_p19_f6a03c19",
   "organism": "hsapiens",
   "sources": {
      "CORUM": {
         "name": "CORUM protein complexes",
         "version": "28.11.2022 Corum 4.1"
      },
      "GO:BP": {
         "name": "biological process",
         "version": "annotations: BioMart\nclasses: releases/2025-03-16"
      },
      "GO:CC": {
         "name": "cellular component",
         "version": "annotations: BioMart\nclasses: releases/2025-03-16"
      },
      "GO:MF": {
         "name": "molecular function",
         "version": "annotations: BioMart\nclasses: releases/2025-03-16"
      },
      "HP": {
         "name": "Human Phenotype Ontology",
         "version": "annotations: 05.2025\nclasses: None"
      },
      "HPA": {
         "name": "Human Protein Atlas",
         "version": "annotations: HPA website: 24-10-19\nclasses: script: 25-02-07"
 

# g:GOSt Endpoint

## Basal-like

In [5]:
# Create the gene-pathway membership network for the group
df_edges, df_nodes = create_pathway_related_network('Basal-like')

In [6]:
# Print the membership edges ordered by q-value
pd.merge(
    left=df_edges,
    right=df_nodes,
    how='inner',
    left_on='target',
    right_on='id'
) \
[['source', 'target', 'label', 'qvalue']] \
.sort_values(by='qvalue', ascending=True, ignore_index=True)

,source,target,label,qvalue
0,CCND2,KEGG:05202,Transcriptional misregulation in cancer,3.206731e-08
1,RXRG,KEGG:05202,Transcriptional misregulation in cancer,3.206731e-08
2,KLF3,KEGG:05202,Transcriptional misregulation in cancer,3.206731e-08
3,ESR1,KEGG:05202,Transcriptional misregulation in cancer,3.206731e-08
4,PAPLN,KEGG:05202,Transcriptional misregulation in cancer,3.206731e-08
...,...,...,...,...
112,HAS2,KEGG:05214,Glioma,4.006518e-02
113,PRDM16,KEGG:05214,Glioma,4.006518e-02
114,CCDC50,KEGG:05214,Glioma,4.006518e-02
115,AFF1,KEGG:05214,Glioma,4.006518e-02


## Luminal A

In [7]:
# Create the gene-pathway membership network for the group
df_edges, df_nodes = create_pathway_related_network('Luminal A')

In [8]:
# Print the membership edges ordered by q-value
pd.merge(
    left=df_edges,
    right=df_nodes,
    how='inner',
    left_on='target',
    right_on='id'
) \
[['source', 'target', 'label', 'qvalue']] \
.sort_values(by='qvalue', ascending=True, ignore_index=True)

,source,target,label,qvalue
0,FLNC,KEGG:04510,Focal adhesion,0.016462
1,LAMA2,KEGG:04510,Focal adhesion,0.016462
2,PDGFRA,KEGG:04510,Focal adhesion,0.016462
3,PRKG1,KEGG:04510,Focal adhesion,0.016462
4,GATA6,KEGG:04510,Focal adhesion,0.016462
5,TCF4,KEGG:04510,Focal adhesion,0.016462
6,DIO2,KEGG:04510,Focal adhesion,0.016462
7,PRRX1,KEGG:04510,Focal adhesion,0.016462
8,TRPM3,KEGG:04510,Focal adhesion,0.016462
9,CROT,KEGG:04510,Focal adhesion,0.016462


## Luminal B

In [9]:
# Create the gene-pathway membership network for the group
df_edges, df_nodes = create_pathway_related_network('Luminal B')

In [10]:
# Print the membership edges ordered by q-value
pd.merge(
    left=df_edges,
    right=df_nodes,
    how='inner',
    left_on='target',
    right_on='id'
) \
[['source', 'target', 'label', 'qvalue']] \
.sort_values(by='qvalue', ascending=True, ignore_index=True)

,source,target,label,qvalue
0,ZBTB4,KEGG:05200,Pathways in cancer,0.000001
1,DZIP1,KEGG:05200,Pathways in cancer,0.000001
2,WAC,KEGG:05200,Pathways in cancer,0.000001
3,DCP2,KEGG:05200,Pathways in cancer,0.000001
4,CBX7,KEGG:05200,Pathways in cancer,0.000001
...,...,...,...,...
388,CALN1,KEGG:04520,Adherens junction,0.046142
389,NAV1,KEGG:04520,Adherens junction,0.046142
390,KCNJ2,KEGG:04520,Adherens junction,0.046142
391,ZC2HC1C,KEGG:04520,Adherens junction,0.046142


## Normal

In [11]:
# Create the gene-pathway membership network for the group
df_edges, df_nodes = create_pathway_related_network('Normal')

In [12]:
# Print the membership edges ordered by q-value
pd.merge(
    left=df_edges,
    right=df_nodes,
    how='inner',
    left_on='target',
    right_on='id'
) \
[['source', 'target', 'label', 'qvalue']] \
.sort_values(by='qvalue', ascending=True, ignore_index=True)

,source,target,label,qvalue
0,NRP1,KEGG:04360,Axon guidance,1.559358e-07
1,GNAI1,KEGG:04360,Axon guidance,1.559358e-07
2,PAK4,KEGG:04360,Axon guidance,1.559358e-07
3,PLXNA4,KEGG:04360,Axon guidance,1.559358e-07
4,CFL2,KEGG:04360,Axon guidance,1.559358e-07
...,...,...,...,...
3866,TSPOAP1,KEGG:05162,Measles,4.974668e-02
3867,NR4A3,KEGG:05162,Measles,4.974668e-02
3868,CD248,KEGG:05162,Measles,4.974668e-02
3869,EPDR1,KEGG:05162,Measles,4.974668e-02
